# Functional programming

Most scientists start programming with **functions**. Sometimes this comes from language history (for example older Fortran), and sometimes simply because classes feel less familiar at first.

> *Definition*: **Functional programming (FP)** emphasises **pure functions**&mdash;functions that return the same output for the same input and avoid side effects.

Pure functions are easier to reason about, test, and combine, which is why FP remains a powerful design style in scientific computing.

Historically, science and engineering languages were largely procedural. Later, object-oriented languages made classes and inheritance mainstream. Today, many scientific codebases use both styles, often in the same project (for example Python).

## Benefits

So far in this course we have focused on classes. Here we shift perspective: FP is also a strong way to structure software.

FP tends to bring:

- **Clarity and predictability**: Functions behave like mappings from input to output.
- **Simpler testing**: Pure functions can be tested in isolation.
- **Composability**: Small functions can be combined into richer behaviour.
- **Fewer state-related bugs**: Less hidden mutation means fewer surprises.
- **Parallel-friendliness**: Stateless functions are often easier to parallelise.

Because most readers are already somewhat familiar with FP, we focus here on three ideas that matter when comparing FP and OOP:

- **(Im)mutability**: In FP, data is often treated as immutable. Instead of changing objects in place, you produce new values. This prevents many bugs caused by unexpected shared-state updates.

- **State**: State is the information a program remembers (variables, attributes, data structures) that can influence behaviour.

- **Purity**: A pure function depends only on inputs and has no side effects (for example printing, file writes, or mutation of external state).

**A Python reality check.**
Python does not enforce purity or immutability. Lists and dictionaries are mutable, and side effects are easy to introduce. That flexibility is useful, but it requires discipline: apply FP principles where they improve clarity and reliability, and allow side effects only where they are genuinely needed.

## FP and OOP

Both paradigms are useful. A practical rule of thumb is:

- Use FP for predictable transformations.
- Use OOP to organise state and behaviour around domain entities.

When combined well, you get clearer structure and easier maintenance.

1. **Reusable functional logic**: Small pure functions can be reused across classes.

1. **Clear separation of concerns**: Keep state in classes; keep transformations in focused functions.

1. **Controlled state changes**: Use mutable state where needed, but keep core logic as pure as practical.

1. **Better readability**: Short, focused methods are easier to understand than large state-heavy methods.

1. **Easier testing**: Pure helpers and stateful wrappers can be tested independently.

1. **Composability**: Complex behaviour can be assembled from small, reusable units.

That is a lot of concepts, so let’s move to concrete examples.

## Pure function in a class

A useful starting point is a small pure function, then wrapping it in a class that carries state. This mirrors a common scientific pattern: keep core maths pure and reusable; use classes only when you need persistent parameters or configuration.

We'll start with a simple power-law helper. We type `x` as a NumPy array because that reflects real scientific Python practice.

In [ ]:
import numpy as np
import numpy.typing as npt


def power_law(x: npt.NDArray, a: float, b: float) -> npt.NDArray:
    """Evaluates a power law

    Args:
        x: Input
        a: Coefficient
        b: Exponent

    Returns:
        Evaluate power law
    """
    return a * x**b

This function is:

1. **Pure**: same input -> same output.
2. **Side-effect free**: it does not mutate external state.
3. **Reusable and testable**: behaviour is simple input/output.

Now let’s use it inside a class.

In [ ]:
class PowerLawModel:
    """A power-law relationship.

    Args:
        a: Coefficient (scaling factor)
        b: Exponent
    """

    def __init__(self, a: float, b: float):
        self.a: float = a
        self.b: float = b

    def evaluate(self, x: npt.NDArray) -> npt.NDArray:
        """Evaluate the model at a given x using the pure function."""
        return power_law(x, self.a, self.b)

We can use the model as follows:

In [ ]:
model: PowerLawModel = PowerLawModel(a=2.0, b=3.0)
y: npt.NDArray = model.evaluate(np.array([4.0, 8.0]))
print(y)

This example shows how FP and OOP complement each other:

- `PowerLawModel` stores state (`a` and `b`).
- `power_law` contains the mathematical logic.
- `evaluate` delegates computation to the pure function.
- The math function can be reused independently of the class.
- You can test function and class behaviour separately.

## Pure function with a class argument

Now let’s reverse the direction: a pure function that *accepts* a class instance and still remains pure.

In [ ]:
def scale_and_sum(model: PowerLawModel, x: npt.NDArray, factor: float) -> float:
    """Multiply the model output by a factor and sum all values.

    This function is pure as long as ``model.evaluate`` is pure and ``model`` is not mutated.
    """
    y: npt.NDArray = model.evaluate(x)

    return np.sum(y * factor)

We can use the ``scale_and_sum`` function as follows:

In [ ]:
x_values: npt.NDArray = np.array([1.0, 2.0, 3.0])
model: PowerLawModel = PowerLawModel(a=2.0, b=3.0)

result: float = scale_and_sum(model, x_values, factor=0.5)
print(result)  # (2*1^3 + 2*2^3 + 2*3^3) * 0.5 = (2 + 16 + 54) * 0.5 = 36.0

Why is `scale_and_sum` still pure?

- It does not mutate `model`.
- It only reads model state via `model.evaluate`.
- Given the same model and input array, it returns the same result.

So this is the mirror image of the previous example:

- **Before**: a stateful class used a pure function.
- **Now**: a pure function consumes a class instance as read-only input.

This pattern is common in scientific software and works well with duck typing: any object that provides `evaluate()` can be used.

Important caveat: not all useful code can be pure. Real systems need side effects (logging, file I/O, state updates). The key is to be explicit about where side effects happen and keep core computations pure where practical.

In short, combining FP and OOP thoughtfully gives you flexibility, composability, and maintainability.

## Exercises

Using `power_law`, `PowerLawModel`, and `scale_and_sum`:

1. Write a new pure function for a meaningful transformation (for example scaling, normalising, or combining arrays).
2. Use it inside a class method without mutating class state.
3. Add a deliberate side effect (for example update an attribute) and compare behaviour with the pure version.
4. Pass class instances into pure functions and compare outputs across different model implementations.

> **Tip:** The main learning goal is to distinguish pure logic from side effects and decide deliberately where each belongs.

Apply to your own project:

5. Identify which parts are best represented as classes and which as functions.
6. Check for duplicated logic and apply DRY where possible.
7. Mark which parts are pure and which involve side effects, so this is clear for future maintainers.

> **Tip:** Treat this as a small design pass. Clarifying structure and purity now makes future refactoring much easier.